In [11]:
from bs4 import BeautifulSoup
import requests
import os


In [13]:
url = "https://www.uninorte.edu.co/web/sobre-nosotros/normatividad"
page = requests.get(url)
soup = BeautifulSoup(page.text, 'html.parser')

In [16]:
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Crear carpeta si no existe
os.makedirs('reglamentos', exist_ok=True)

# Configurar sesión con reintentos
session = requests.Session()
retry = Retry(connect=3, backoff_factor=0.5)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

# Headers para simular navegador
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

# Buscar el div con clase "c_cr"
div_reglamentos = soup.find('div', class_='c_cr')

if div_reglamentos:
    # Encontrar todos los enlaces a PDFs dentro de ese div
    for link in div_reglamentos.find_all('a', href=True):
        href = link['href']
        if href.endswith('.pdf'):
            try:
                # Validar que sea URL completa
                if not href.startswith('http'):
                    href = url.split('/web')[0] + href
                
                # Descargar con timeout
                pdf_response = session.get(href, headers=headers, timeout=10)
                pdf_response.raise_for_status()
                
                filename = href.split('/')[-1]
                with open(f'reglamentos/{filename}', 'wb') as f:
                    f.write(pdf_response.content)
                print(f"✓ Descargado: {filename}")
                
                # Pausa entre descargas para no saturar el servidor
                time.sleep(1)
                
            except Exception as e:
                print(f"✗ Error descargando {href}: {e}")
                continue
else:
    print("No se encontró el div con clase 'c_cr'")


✓ Descargado: Reg_profesor_%20junio_2021.pdf
✓ Descargado: reglamento_Interno_Trabajo_Uninorte.pdf
✓ Descargado: Reglamento%20de%20Egresados%20Uninorte.pdf
✓ Descargado: Res_Con_Aca_No_32.pdf
✓ Descargado: Res_Rect_No_35_Pol_Derechos_Humanos_Un.pdf
✓ Descargado: reglamento_propiedad_intelectual.pdf
✓ Descargado: Res_Rec_No_66_protocolo_uninorte.pdf
✓ Descargado: RES_92_20120001.pdf
✓ Descargado: nrr19.pdf
✓ Descargado: Res_Rect_N_Est_Bienestar_Un.pdf
